# Data Loading and Validation

Before any analysis, this notebook checks that the data can be trusted. It reads the SQLite database built by `src/load_data.py` (raw table, loaded unmodified) and `sql/01` to `sql/02b` (clean tables), and shows what was wrong with the raw data and how each problem was handled. Every cleaning decision is documented in `reports/profiling_notes.md`.

To rebuild the database from scratch: run `python src/load_data.py`, then `sql/01_create_tables.sql`, `sql/02_clean_data.sql` and `sql/02b_load_tables.sql` with `src/run_sql_file.py`, in that order.

## 1. Setup

Connects to the database and lists its tables and views. The raw table `raw_online_retail` is never edited. `customers`, `products` and `orders` are built from the `clean_lines` view, which holds all cleaning rules.

In [10]:
import sqlite3

import pandas as pd

DB_PATH = "../data/processed/retail.db"   # notebooks/ sits one level below the project root
conn = sqlite3.connect(DB_PATH)

def q(sql):
    """Run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)

print(q("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view')"))



                name   type
0  raw_online_retail  table
1          customers  table
2           products  table
3             orders  table
4        clean_lines   view


## 2. Raw table: size and period

The raw table holds 541,909 invoice lines across 25,900 invoices and 4,372 customers, from 1 December 2010 to 9 December 2011. This matches the source file row for row. The last timestamp (9 December 2011, 12:50) sets the reference date used for days since last order in the churn analysis.

In [11]:
q("""SELECT COUNT(*)                   AS total_rows,
            COUNT(DISTINCT InvoiceNo)  AS invoices,
            COUNT(DISTINCT CustomerID) AS customers,
            MIN(InvoiceDate)           AS first_date,
            MAX(InvoiceDate)           AS last_date
     FROM raw_online_retail""")

,total_rows,invoices,customers,first_date,last_date
0,541909,25900,4372,2010-12-01 08:26:00,2011-12-09 12:50:00


## 3. Missing customer IDs and country labels

135,080 rows (24.9%) have no `CustomerID`. They cannot be analysed per customer, so they are excluded from all customer-level results.

The raw data has 38 country labels. `EIRE` is Ireland (3 customers) and `RSA` is South Africa (1 customer), and both are relabelled in the `customers` table. `Unspecified` (4 customers) and `European Community` (1 customer) are not real countries, so they are kept in the data but left out of the country chart. Hong Kong has 288 rows but no customer IDs, so it disappears from country revenue.

8 customers appear under two countries. Each is assigned the country with the most distinct invoices, ties going to the most recent invoice, then alphabetical order.

In [12]:
print(q("""SELECT SUM(CustomerID IS NULL)                                  AS rows_without_id,
              ROUND(100.0 * SUM(CustomerID IS NULL) / COUNT(*), 2)     AS pct_of_rows
       FROM raw_online_retail""").to_string(index=False))

print(q("""SELECT COUNT(DISTINCT Country) AS country_labels FROM raw_online_retail""").to_string(index=False))

print(q("""SELECT Country, COUNT(*) AS n_rows, COUNT(DISTINCT CustomerID) AS customers
       FROM raw_online_retail
       WHERE Country IN ('EIRE', 'RSA', 'Hong Kong', 'Unspecified', 'European Community')
       GROUP BY Country ORDER BY n_rows DESC""").to_string(index=False))

print(q("""SELECT COUNT(*) AS customers_with_multiple_countries
       FROM (SELECT CustomerID FROM raw_online_retail
             WHERE CustomerID IS NOT NULL
             GROUP BY CustomerID HAVING COUNT(DISTINCT Country) > 1)""").to_string(index=False))

 rows_without_id  pct_of_rows
          135080        24.93
 country_labels
             38
           Country  n_rows  customers
              EIRE    8196          3
       Unspecified     446          4
         Hong Kong     288          0
European Community      61          1
               RSA      58          1
 customers_with_multiple_countries
                                 8


## 4. Price and quantity extremes

Prices range from -£11,062.06 to £38,970.00 and quantities from -80,995 to 80,995. The extreme rows are not normal sales:

- **Invoice `C556445`:** a cancelled "Manual" entry (StockCode `M`) priced at £38,970, an accounting adjustment and not a product.
- **Invoices `A563186` and `A563187`:** "Adjust bad debt" entries at -£11,062.06 with no customer ID.
- **Invoice `581483` and cancellation `C581484`:** an order of 80,995 units for customer 16446 and its cancellation 12 minutes later. The order was never a real sale, so it is excluded by name.

There are 2 negative prices and 2,515 zero prices. All are removed by the `UnitPrice > 0` rule (40 of the zero-price rows belong to identified customers).

In [13]:
print(q("""SELECT MIN(UnitPrice) AS min_price, MAX(UnitPrice) AS max_price,
              MIN(Quantity) AS min_qty, MAX(Quantity) AS max_qty,
              SUM(UnitPrice < 0) AS negative_price, SUM(UnitPrice = 0) AS zero_price
       FROM raw_online_retail""").to_string(index=False))

q("""SELECT InvoiceNo, StockCode, Description, Quantity, UnitPrice, CustomerID
     FROM raw_online_retail
     WHERE UnitPrice = (SELECT MIN(UnitPrice) FROM raw_online_retail)
        OR UnitPrice = (SELECT MAX(UnitPrice) FROM raw_online_retail)
        OR ABS(Quantity) = 80995""")

 min_price  max_price  min_qty  max_qty  negative_price  zero_price
 -11062.06    38970.0   -80995    80995               2        2515


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
0,C556445,M,Manual,-1,38970.00,15098.0
1,A563186,B,Adjust bad debt,1,-11062.06,NaN
2,A563187,B,Adjust bad debt,1,-11062.06,NaN
3,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446.0
4,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446.0


## 5. Cancellations

There are 9,288 cancellation invoices (prefix `C`), and all have negative quantities. Another 1,336 rows have negative quantities without the prefix. All 1,336 have zero price and no customer ID, so they are stock adjustments (damaged, lost, "check") that the price and customer rules already remove. Both rules are needed, but the second adds nothing on this data.

Cancellations are dropped, but a cancellation does not always mean the sale was unreal:

- **Return after a real sale:** the purchase still counts as activity. Only the `C` row is dropped.
- **Cancel-and-reissue:** the original invoice was voided and replaced minutes later, so keeping both would count the goods twice. Customer 15749 is an example: invoices `540815` and `540818` (£22,998.40) were voided by `C550456` and reissued as `550461` (£21,535.90). The voided originals are removed.

The audit covered the customers with the largest cancelled value. Voided originals were removed for 12346, 16446, 15749 and 16029, and 12931 was classified as ordinary returns. Smaller reissues may remain.

In [14]:
print(q("""SELECT CASE WHEN InvoiceNo LIKE 'C%' THEN 'C invoice' ELSE 'not C' END AS invoice_type,
              CASE WHEN Quantity < 0 THEN 'negative qty' ELSE 'positive qty' END AS qty_type,
              COUNT(*) AS n_rows
       FROM raw_online_retail
       GROUP BY invoice_type, qty_type""").to_string(index=False))

print(q("""SELECT COUNT(*) AS n_rows, SUM(UnitPrice = 0) AS zero_price, SUM(CustomerID IS NULL) AS no_id
       FROM raw_online_retail
       WHERE Quantity < 0 AND InvoiceNo NOT LIKE 'C%'""").to_string(index=False))

q("""SELECT InvoiceNo, InvoiceDate, COUNT(*) AS n_lines, ROUND(SUM(Quantity * UnitPrice), 2) AS value
     FROM raw_online_retail
     WHERE InvoiceNo IN ('540815','540818','C550456','550461')
     GROUP BY InvoiceNo, InvoiceDate ORDER BY InvoiceDate""")

invoice_type     qty_type  n_rows
   C invoice negative qty    9288
       not C negative qty    1336
       not C positive qty  531285
 n_rows  zero_price  no_id
   1336        1336   1336


,InvoiceNo,InvoiceDate,n_lines,value
0,540815,2011-01-11 12:55:00,3,15160.9
1,540818,2011-01-11 12:57:00,2,7837.5
2,C550456,2011-04-18 13:08:00,5,-22998.4
3,550461,2011-04-18 13:20:00,5,21535.9


## 6. Non-product stock codes

Fees, postage and adjustments are recorded as stock codes. Across the raw table, ten codes are not merchandise:

| Code | Meaning | Rows | Net revenue |
|---|---|---:|---:|
| POST | Postage | 1,256 | £66,231 |
| DOT | Dotcom postage | 710 | £206,245 |
| M | Manual entry | 572 | -£68,672 |
| C2 | Carriage | 144 | £6,986 |
| D | Discount | 77 | -£5,696 |
| S | Samples | 63 |

In [15]:
q("""SELECT UPPER(StockCode) AS code, COUNT(*) AS n_rows,
            ROUND(SUM(Quantity * UnitPrice), 2) AS net_revenue
     FROM raw_online_retail
     WHERE UPPER(StockCode) IN ('POST','DOT','M','C2','D','S','BANK CHARGES','AMAZONFEE','CRUK','B')
     GROUP BY UPPER(StockCode)
     ORDER BY n_rows DESC""")

,code,n_rows,net_revenue
0,POST,1256,66230.64
1,DOT,710,206245.48
2,M,572,-68671.64
3,C2,144,6986.00
4,D,77,-5696.22
5,S,63,-3049.39
6,BANK CHARGES,37,-7175.64
7,AMAZONFEE,34,-221520.50
8,CRUK,16,-7933.43
9,B,3,-11062.06


## 7. Cleaning waterfall

All cleaning rules live in one SQL view, `clean_lines` (`sql/02_clean_data.sql`). Applying them one after another shows how many rows each removes:

| Rule | Rows removed | Rows left |
|---|---:|---:|
| Raw | | 541,909 |
| 1. No CustomerID | 135,080 | 406,829 |
| 2. Cancellation invoices | 8,905 | 397,924 |
| 3. Quantity > 0 | 0 | 397,924 |
| 4. UnitPrice > 0 | 40 | 397,884 |
| 5. Fees and postage | 1,544 | 396,340 |
| 6. Voided originals | 19 | 396,321 |
| **Total** | **145,588** | |

Rule 3 removes nothing because every negative quantity is already caught by rules 1 and 2. It stays as a safety guard. Rule 6 lists six invoices: 541431, 581483, 540815, 540818, 570094 and 570097. The row count after the last rule equals the row count of `clean_lines`.

39 of the 4,372 raw customers (0.9%) have no valid purchase left and drop out: 33 had only cancellations, 4 had only a fee line, 1 had only a zero-price row, and 1 (12346) had its only order cancelled in full.

In [16]:
total = conn.execute("SELECT COUNT(*) FROM raw_online_retail").fetchone()[0]

rules = [
    ("1. No CustomerID",          "CustomerID IS NOT NULL"),
    ("2. Cancellation invoices",  "InvoiceNo NOT LIKE 'C%'"),
    ("3. Quantity > 0",           "Quantity > 0"),
    ("4. UnitPrice > 0",          "UnitPrice > 0"),
    ("5. Fees and postage",       "UPPER(StockCode) NOT IN ('POST','DOT','M','C2','D','S','BANK CHARGES','AMAZONFEE','CRUK','B')"),
    ("6. Voided originals",       "InvoiceNo NOT IN ('541431','581483','540815','540818','570094','570097')"),
]

rows, conds, prev = [("Raw", total, 0)], [], total
for name, cond in rules:
    conds.append(cond)
    left = conn.execute("SELECT COUNT(*) FROM raw_online_retail WHERE " + " AND ".join(conds)).fetchone()[0]
    rows.append((name, left, prev - left))
    prev = left

waterfall = pd.DataFrame(rows, columns=["rule", "rows_left", "rows_removed"])
print(waterfall.to_string(index=False))
print("Total removed:", total - prev)
print("Matches clean_lines:", prev == conn.execute("SELECT COUNT(*) FROM clean_lines").fetchone()[0])

                    rule  rows_left  rows_removed
                     Raw     541909             0
        1. No CustomerID     406829        135080
2. Cancellation invoices     397924          8905
         3. Quantity > 0     397924             0
        4. UnitPrice > 0     397884            40
     5. Fees and postage     396340          1544
     6. Voided originals     396321            19
Total removed: 145588
Matches clean_lines: True


## 8. Final reconciliation

The three clean tables are built from `clean_lines`, so they should agree with it and with each other.

| Check | Result |
|---|---|
| Rows in `clean_lines` and `orders` | 396,321 and 396,321 |
| Customers / products / invoices | 4,333 / 3,659 / 18,396 |
| Revenue in `orders` | £8,480,598.41 |
| Revenue joined through `customers` | £8,480,598.41 |
| Revenue joined through `products` | £8,480,598.41 |
| Orders without a customer / without a product | 0 / 0 |

Revenue is identical whichever table it is joined through, so no join drops or duplicates rows. Revenue means merchandise revenue, excluding postage and fees.

In [17]:
print(q("""SELECT (SELECT COUNT(*) FROM clean_lines)                      AS clean_lines_rows,
              (SELECT COUNT(*) FROM orders)                           AS orders_rows,
              (SELECT COUNT(*) FROM customers)                        AS customers,
              (SELECT COUNT(*) FROM products)                         AS products,
              (SELECT COUNT(DISTINCT InvoiceNo) FROM orders)          AS invoices""").to_string(index=False))

print(q("""SELECT (SELECT ROUND(SUM(Quantity * UnitPrice), 2) FROM orders) AS revenue_orders,
              (SELECT ROUND(SUM(o.Quantity * o.UnitPrice), 2)
               FROM orders o JOIN customers c ON c.CustomerID = o.CustomerID) AS revenue_via_customers,
              (SELECT ROUND(SUM(o.Quantity * o.UnitPrice), 2)
               FROM orders o JOIN products p ON p.StockCode = o.StockCode)    AS revenue_via_products""").to_string(index=False))

print(q("""SELECT (SELECT COUNT(*) FROM orders WHERE CustomerID NOT IN (SELECT CustomerID FROM customers)) AS orders_without_customer,
              (SELECT COUNT(*) FROM orders WHERE StockCode NOT IN (SELECT StockCode FROM products))      AS orders_without_product""").to_string(index=False))

 clean_lines_rows  orders_rows  customers  products  invoices
           396321       396321       4333      3659     18396
 revenue_orders  revenue_via_customers  revenue_via_products
     8480598.41             8480598.41            8480598.41
 orders_without_customer  orders_without_product
                       0                       0


## Conclusion

The raw data had four serious problems: a quarter of rows without a customer ID, cancellations (including cancel-and-reissue pairs), fees and postage recorded as products, and extreme adjustment rows. All are handled by six documented rules, and the clean tables reconcile exactly. The analysis continues in `02_churn_analysis.ipynb`. Every decision is logged in `reports/profiling_notes.md`.